# 01 — Data Acquisition from CZ CELLxGENE Census

This notebook downloads and caches pseudobulk expression data from the
[CZ CELLxGENE Census](https://cellxgene.cziscience.com/), the most
curated aggregation of human single-cell RNA-seq datasets.

**Outputs:**
- `data/processed/pseudobulk_mean_cpm.parquet` — Mean CPM per gene per cell type
- `data/processed/pseudobulk_detection_rate.parquet` — Fraction of cells expressing each gene
- `data/processed/obs_metadata.parquet` — Cell-level metadata (for reference)
- `data/processed/cell_type_summary.parquet` — Cell type counts and dataset info

**Runtime:** ~20-60 min depending on network speed (queries ~100+ cell types).
Cached results are reused on subsequent runs.

In [1]:
import sys
from pathlib import Path
import logging

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")
logger = logging.getLogger("01_data_acquisition")

In [2]:
from src.data_loader import (
    get_census_version,
    open_census,
    get_obs_metadata,
    get_var_metadata,
    filter_adult_cells,
    get_cell_type_summary,
    compute_pseudobulk,
    save_pseudobulk,
    load_pseudobulk,
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Check Census Version & Skip if Cached

If pseudobulk data already exists, we skip the download.

In [3]:
CACHED_MEAN = DATA_DIR / "pseudobulk_mean_cpm.parquet"
CACHED_DET = DATA_DIR / "pseudobulk_detection_rate.parquet"

if CACHED_MEAN.exists() and CACHED_DET.exists():
    print("Cached pseudobulk data found! Loading...")
    mean_df, det_df = load_pseudobulk(DATA_DIR)
    print(f"Loaded: {mean_df.shape[0]:,} genes × {mean_df.shape[1]} cell types")
    SKIP_DOWNLOAD = True
else:
    print("No cached data — will download from Census.")
    SKIP_DOWNLOAD = False

No cached data — will download from Census.


## 2. Connect to Census & Explore Metadata

Query cell metadata for healthy (disease = 'normal'), primary human tissue.

In [4]:
if not SKIP_DOWNLOAD:
    version = get_census_version()
    print(f"Census version: {version}")
    census = open_census(census_version="stable")

Census version: 2025-11-08


The "stable" release is currently 2025-11-08. Specify 'census_version="2025-11-08"' in future calls to open_soma() to ensure data consistency.
2026-03-05 20:43:59,141 | cellxgene_census | The "stable" release is currently 2025-11-08. Specify 'census_version="2025-11-08"' in future calls to open_soma() to ensure data consistency.


In [5]:
if not SKIP_DOWNLOAD:
    obs_df = get_obs_metadata(census)
    print(f"Total cells (healthy, primary): {len(obs_df):,}")
    print(f"\nDevelopment stages:\n{obs_df['development_stage'].value_counts().head(15)}")
    print(f"\nTissues (general):\n{obs_df['tissue_general'].value_counts().head(20)}")

2026-03-05 20:44:04,895 | src.data_loader | Querying obs metadata with filter: disease == 'normal' and is_primary_data == True
2026-03-05 20:44:16,735 | src.data_loader | Retrieved 62,464,283 cells


Total cells (healthy, primary): 62,464,283

Development stages:
development_stage
unknown                               2740934
15th week post-fertilization stage    2131676
29-year-old stage                     2028985
50-year-old stage                     1826490
42-year-old stage                     1520350
adult stage                           1286376
12th week post-fertilization stage    1168441
sixth decade stage                    1146695
17th week post-fertilization stage    1131076
80 year-old and over stage            1127714
60-year-old stage                     1119096
seventh decade stage                  1046249
56-year-old stage                     1035846
61-year-old stage                      955395
30-year-old stage                      925878
Name: count, dtype: int64

Tissues (general):
tissue_general
brain                     18539831
blood                     13822662
eye                        7806560
lung                       3148828
breast                     

## 3. Filter to Adult Cells

Exclude fetal, embryonic, pediatric, organoid, and cell-culture samples.

In [6]:
if not SKIP_DOWNLOAD:
    obs_adult = filter_adult_cells(obs_df)
    print(f"Adult cells remaining: {len(obs_adult):,}")
    print(f"\nTop development stages after filtering:")
    print(obs_adult["development_stage"].value_counts().head(10))

2026-03-05 20:44:22,501 | src.data_loader | Filtered to 59,743,744 adult cells (removed 2,720,539 fetal/excluded)


Adult cells remaining: 59,743,744

Top development stages after filtering:
development_stage
unknown                               2539639
15th week post-fertilization stage    2131676
29-year-old stage                     2028985
50-year-old stage                     1826490
42-year-old stage                     1520350
adult stage                           1286376
12th week post-fertilization stage    1168441
sixth decade stage                    1146695
17th week post-fertilization stage    1131076
80 year-old and over stage            1127714
Name: count, dtype: int64


## 4. Cell Type Summary & Selection

Require ≥500 cells and ≥2 independent datasets per cell type.

In [8]:
if not SKIP_DOWNLOAD:
    ct_summary = get_cell_type_summary(
        obs_adult, min_cells=500, min_datasets=2
    )
    print(f"Cell types passing thresholds: {len(ct_summary)}")
    print(f"\nTop 30 cell types by cell count:")
    display(ct_summary.head(30))


/Users/LMWee/Desktop/Specific_Transcript/src/data_loader.py:194: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["cell_type", "cell_type_ontology_term_id"])
2026-03-05 20:50:35,008 | src.data_loader | Cell types passing thresholds (>=500 cells, >=2 datasets): 530 / 806404


Cell types passing thresholds: 530

Top 30 cell types by cell count:


,cell_type,cell_type_ontology_term_id,n_cells,n_datasets,tissues
0,neuron,CL:0000540,3168735,134,"[adrenal gland, brain, central nervous system,..."
1,oligodendrocyte,CL:0000128,2514375,151,"[brain, central nervous system, cortex, eye, s..."
2,"naive thymus-derived CD4-positive, alpha-beta ...",CL:0000895,2387978,37,"[adipose tissue, adrenal gland, blood, bone ma..."
3,fibroblast,CL:0000057,1991224,194,"[adipose tissue, adrenal gland, bladder organ,..."
4,glutamatergic neuron,CL:0000679,1886071,26,"[brain, central nervous system, spinal cord]"
5,"central memory CD4-positive, alpha-beta T cell",CL:0000904,1553605,18,"[blood, bone marrow, colon, immune system, liv..."
6,L2/3-6 intratelencephalic projecting glutamate...,CL:4023040,1496676,17,[brain]
7,unknown,unknown,1455983,81,"[adipose tissue, adrenal gland, blood, bone ma..."
8,astrocyte,CL:0000127,1344962,146,"[brain, central nervous system, eye, spinal cord]"
9,retinal rod cell,CL:0000604,1303969,15,"[central nervous system, eye]"


In [9]:
if not SKIP_DOWNLOAD:
    # Use top 100 cell types by cell count (sufficient for comprehensive analysis)
    cell_types = ct_summary["cell_type"].head(100).tolist()
    print(f"\nWill compute pseudobulk for {len(cell_types)} cell types.")
    print("First 20:", cell_types[:20])



Will compute pseudobulk for 100 cell types.
First 20: ['neuron', 'oligodendrocyte', 'naive thymus-derived CD4-positive, alpha-beta T cell', 'fibroblast', 'glutamatergic neuron', 'central memory CD4-positive, alpha-beta T cell', 'L2/3-6 intratelencephalic projecting glutamatergic neuron', 'unknown', 'astrocyte', 'retinal rod cell', 'CD14-positive, CD16-negative classical monocyte', 'CD16-positive, CD56-dim natural killer cell, human', 'naive B cell', 'naive thymus-derived CD8-positive, alpha-beta T cell', 'macrophage', 'effector memory CD4-positive, alpha-beta T cell', 'natural killer cell', 'effector memory CD8-positive, alpha-beta T cell', 'CD4-positive, alpha-beta T cell', 'epithelial cell']


In [12]:
if not SKIP_DOWNLOAD:
    CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints"
    
    mean_df, det_df = compute_pseudobulk(
        census,
        obs_adult,
        cell_types,
        max_cells_per_type=10_000,
        random_seed=42,
        checkpoint_dir=CHECKPOINT_DIR,  # Save progress every 10 cell types
        checkpoint_every=10,
    )
    print(f"\nPseudobulk matrix: {mean_df.shape[0]:,} genes × {mean_df.shape[1]} cell types")


2026-03-06 18:07:38,117 | src.data_loader | Resuming from checkpoint: /Users/LMWee/Desktop/Specific_Transcript/data/checkpoints/checkpoint_60.pkl
2026-03-06 18:07:38,185 | src.data_loader | Resuming from cell type 61/100
2026-03-06 18:07:38,186 | src.data_loader | [61/100] Processing: hematopoietic multipotent progenitor cell
2026-03-06 18:07:38,317 | src.data_loader |   Subsampled to 10000 cells
/Users/LMWee/Desktop/Specific_Transcript/src/data_loader.py:303: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  adata = cellxgene_census.get_anndata(
/opt/anaconda3/lib/python3.12/functools.py:907: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/opt/anaconda3/lib/python3.12/functools.py:907: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-03-06 18:10:53,897 | s


Pseudobulk matrix: 61,497 genes × 100 cell types


## 5. Compute Pseudobulk Expression

For each cell type:
1. Retrieve raw counts (subsampled to ≤10,000 cells for memory).
2. CPM-normalize each cell.
3. Compute **mean CPM** and **detection rate** (fraction of cells with count > 0).

This is the most time-consuming step (~30-60 min for 100+ cell types).

In [ ]:
if not SKIP_DOWNLOAD:
    mean_df, det_df = compute_pseudobulk(
        census,
        obs_adult,
        cell_types,
        max_cells_per_type=10_000,
        random_seed=42,
        checkpoint_dir=DATA_DIR / "checkpoints",
    )
    print(f"\nPseudobulk matrix: {mean_df.shape[0]:,} genes × {mean_df.shape[1]} cell types")

2026-03-07 01:08:56,943 | src.data_loader | [1/100] Processing: neuron
2026-03-07 01:08:57,228 | src.data_loader |   Subsampled to 10000 cells
/Users/LMWee/Desktop/Specific_Transcript/src/data_loader.py:303: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  adata = cellxgene_census.get_anndata(
/opt/anaconda3/lib/python3.12/functools.py:907: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/opt/anaconda3/lib/python3.12/functools.py:907: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-03-07 01:45:48,379 | src.data_loader |   Done — 10000 cells, 41125 genes detected (>0 in ≥1 cell)
2026-03-07 01:45:48,381 | src.data_loader | [2/100] Processing: oligodendrocyte
2026-03-07 01:45:48,581 | src.data_loader |   Subsampled to 10000 cells
/Users/LMWee/Desktop/Specific

In [ ]:
# Try to salvage partial results from interrupted cell
try:
    if 'mean_expressions' in dir():
        print(f"✓ Found {len(mean_expressions)} processed cell types!")
        
        import pickle
        CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints"
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        
        checkpoint_path = CHECKPOINT_DIR / f"checkpoint_{len(mean_expressions)}.pkl"
        with open(checkpoint_path, "wb") as f:
            pickle.dump({
                "mean_expressions": mean_expressions,
                "detection_rates": detection_rates,
                "gene_names": gene_names,
                "gene_ids": gene_ids,
                "processed_idx": len(mean_expressions),
                "cell_types": list(mean_expressions.keys()),
            }, f)
        print(f"✓ Checkpoint saved: {checkpoint_path}")
    else:
        print("✗ No partial data found in memory")
except NameError:
    print("✗ Variables not accessible - data was lost")
except Exception as e:
    print(f"✗ Error: {e}")


✗ No partial data found in memory


## 6. Save Results

In [ ]:
if not SKIP_DOWNLOAD:
    save_pseudobulk(
        mean_df, det_df, DATA_DIR,
        obs_df=obs_adult,
        cell_type_summary=ct_summary,
    )
    census.close()
    print("Done! Data saved to:", DATA_DIR)

## 7. Quick Sanity Check

Verify known marker genes are highly expressed in the expected cell types.

In [ ]:
KNOWN_MARKERS = {
    "ALB": "hepatocyte",
    "CD3D": "T cell",
    "SNAP25": "neuron",
    "TNNT2": "cardiomyocyte",
    "KRT14": "keratinocyte",
    "CD79A": "B cell",
    "PECAM1": "endothelial cell",
}

print("Sanity check — known markers:")
print(f"{'Gene':<10} {'Expected Cell Type':<25} {'Top Cell Type in Data':<30} {'Match?'}")
print("-" * 75)

for gene, expected_ct in KNOWN_MARKERS.items():
    if gene in mean_df.index:
        top_ct = mean_df.loc[gene].idxmax()
        match = expected_ct.lower() in top_ct.lower()
        print(f"{gene:<10} {expected_ct:<25} {top_ct:<30} {'YES' if match else 'CHECK'}")
    else:
        print(f"{gene:<10} {expected_ct:<25} {'NOT FOUND':<30}")

## Summary

Data acquisition complete. Next step: **Notebook 02 — Preprocessing**.